# Local LLM Lab

Run a real open-source LLM on your own laptop, look inside it to understand how it works, then
experiment with fine-tuning. The model is **Qwen2.5-3B-Instruct** — a capable ~3B-parameter open model
that fits on 16 GB of RAM.

**Honest expectations.** This is a learning lab, not a replacement for hosted frontier models. A 3B model
on a CPU will answer basic questions, summarise, and chat — but slowly, and it will sometimes be
confidently wrong. Seeing *where* it breaks is part of the learning.

### The three paths
1. **Understand** (this notebook, `transformers`) — slow, but lets you see the tokenizer, the architecture,
   the raw next-token probabilities, and the generation loop.
2. **Chat fast** (Ollama) — a quantized build that's responsive enough for everyday use (cell near the end).
3. **Fine-tune** (Colab) — train a small LoRA adapter on a free GPU, then run it here.

Run the cells top to bottom. The first model-load cell downloads ~6 GB (cached afterwards).

## 1. Transformer vs RNN (clearing up a common mix-up)

Transformers are **not** a kind of RNN — they *replaced* RNNs for language.

- An **RNN** reads tokens one at a time, carrying a hidden "memory" vector forward. Sequential, hard to
  parallelise, and it tends to forget things from far back.
- A **Transformer** sees all tokens at once and uses **attention**: every token directly looks at every
  other token and decides how much each one matters. That parallelism + direct long-range access is why
  Transformers scaled into modern LLMs and RNNs didn't.

Everything below is a Transformer (Qwen2.5).

## 2. Environment check

Confirm we're on CPU, see how much RAM we have, and set the thread count. (No NVIDIA GPU here, so
`device` will be `cpu` — that's expected.)

In [ ]:
import os, platform
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # the model we use everywhere below

print("Python    :", platform.python_version())
print("PyTorch   :", torch.__version__)
print("CUDA avail:", torch.cuda.is_available(), "(False is expected on this laptop)")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device    :", device)
print("CPU cores :", os.cpu_count())

try:
    import psutil
    gb = psutil.virtual_memory().total / 1e9
    print(f"Total RAM : {gb:.1f} GB")
except Exception:
    pass

# CPU inference benefits from a sensible thread count. Tune if you like.
torch.set_num_threads(max(1, (os.cpu_count() or 4) // 2))
print("Torch threads:", torch.get_num_threads())

## 3. The model: Qwen2.5-3B-Instruct

Qwen2.5 is a family of open models from Alibaba Cloud. The **3B Instruct** variant is small enough to run
here and is *instruction-tuned* (it's been trained to follow chat-style prompts). It's **ungated** — no
license signup needed to download it.

## 4. The tokenizer — how text becomes numbers

A model never sees text; it sees integer **token IDs**. The tokenizer splits text into subword pieces and
maps each to an ID. Let's load it and look.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)

text = "Hello! How do transformers actually work?"
ids = tok(text)["input_ids"]

print("Text       :", text)
print("Token IDs  :", ids)
print("Tokens     :", tok.convert_ids_to_tokens(ids))
print("Round-trip :", tok.decode(ids))
print("Vocab size :", tok.vocab_size)

# Notice how common words are one token but rarer/sub-word chunks split apart.
# The leading characters (e.g. 'Ġ') mark a preceding space in this tokenizer.

## 5. Load the model and look at its architecture

We load in **bfloat16** (~6 GB) so it fits comfortably in 16 GB RAM. The first run downloads the weights
from Hugging Face (~6 GB) — be patient; it's cached after that.

Then we print the config and the module tree so you can *see* the Transformer: a stack of decoder layers,
each with self-attention + an MLP.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
model.eval()

cfg = model.config
print("hidden size      :", cfg.hidden_size)
print("num layers       :", cfg.num_hidden_layers)
print("attention heads  :", cfg.num_attention_heads)
print("KV heads (GQA)    :", getattr(cfg, "num_key_value_heads", "n/a"))
print("vocab size       :", cfg.vocab_size)

n_params = sum(p.numel() for p in model.parameters())
print(f"\ntotal parameters : {n_params/1e9:.2f} B")

# The module tree: embedding -> N decoder layers (attention + MLP) -> norm -> lm_head
print()
print(model)

## 6. How generation works: one token at a time

An LLM is just a **next-token predictor**. Given a sequence, it outputs a score (logit) for every token in
the vocabulary; softmax turns those into probabilities. To generate text, you pick a token, append it, and
repeat.

Let's do a single forward pass and look at the model's top candidates for the *next* token.

In [ ]:
prompt = "The capital of France is"
inputs = tok(prompt, return_tensors="pt")

with torch.no_grad():
    out = model(**inputs)

# logits for the position right after the prompt
next_logits = out.logits[0, -1]
probs = torch.softmax(next_logits.float(), dim=-1)
topk = torch.topk(probs, 10)

print(f"Prompt: {prompt!r}\nTop-10 next-token candidates:\n")
for score, idx in zip(topk.values, topk.indices):
    print(f"  {tok.decode([idx])!r:>15}  p={score.item():.4f}")

## 7. Full generation

Now we let the model generate a full answer with `model.generate`. We use the **chat template** so the
prompt is formatted exactly the way the instruct model expects.

> On CPU this runs at only a few tokens/sec, so we keep `max_new_tokens` small while exploring. This cell
> may take a little while — that's normal.

In [ ]:
messages = [{"role": "user", "content": "In one sentence, what is a transformer in machine learning?"}]
# return_dict=True gives us input_ids + attention_mask (a BatchEncoding) — the form generate() wants.
inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                 return_tensors="pt", return_dict=True)

with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=60, do_sample=False)

# Slice off the prompt tokens so we only decode the newly generated answer.
answer = tok.decode(gen[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

## 8. A small multi-turn chat helper

Wrapping the above in a function gives us a basic conversation loop that keeps history. (Each turn
re-processes the whole history, so keep conversations short on CPU.)

In [ ]:
def chat(history, user_msg, max_new_tokens=128):
    """Append a user turn, generate a reply, return (new_history, reply)."""
    history = history + [{"role": "user", "content": user_msg}]
    inputs = tok.apply_chat_template(history, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = tok.decode(gen[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return history + [{"role": "assistant", "content": reply}], reply

history = []
history, reply = chat(history, "Hi! In one short sentence, what can you help me with?")
print("Assistant:", reply)

history, reply = chat(history, "What did I just ask you?")
print("\nAssistant:", reply)

## 9. The fast path: Ollama

`transformers` on CPU is great for *looking inside* the model, but slow for everyday chatting. For that,
use **[Ollama](https://ollama.com)**, which runs a 4-bit quantized build (~2 GB) much faster.

1. Install Ollama from <https://ollama.com/download>.
2. In a terminal: `ollama run qwen2.5:3b` (first run downloads the model).
3. Ollama serves an HTTP API on `localhost:11434`. The cell below uses it, and skips gracefully if Ollama
   isn't installed/running.

In [ ]:
import requests

def ollama_chat(prompt, model="qwen2.5:3b"):
    try:
        r = requests.post(
            "http://localhost:11434/api/chat",
            json={"model": model,
                  "messages": [{"role": "user", "content": prompt}],
                  "stream": False},
            timeout=180,
        )
        r.raise_for_status()
        return r.json()["message"]["content"]
    except Exception as e:
        return ("(Ollama not reachable: %s)\n"
                "Install from https://ollama.com and run:  ollama run qwen2.5:3b" % e)

print(ollama_chat("Give me one fun fact about octopuses."))

## 10. Fine-tuning — the idea

**Fine-tuning** nudges the model's behaviour using your own examples. We use **LoRA** (Low-Rank
Adaptation): the big model stays frozen and we train a tiny set of extra weights "on top". The resulting
**adapter** is only a few MB.

Two key points:
- **It teaches style/tone/format far more reliably than new facts.** Our example dataset teaches the model
  to always finish with a `TL;DR:` line.
- **Training needs a GPU**, which this laptop doesn't have — so we train on a **free Google Colab GPU**
  (QLoRA), then run the adapter here on CPU. See `finetune/README.md`.

## 11. Build your fine-tuning dataset

Each training example is a `{"messages": [...]}` record with a user turn and the assistant reply you *wish*
the model would give. Use `add_example(...)` to append to `finetune/data/train.jsonl`, then preview it.

In [ ]:
import json

DATA_PATH = "../finetune/data/train.jsonl"   # relative to this notebook (notebooks/)

def add_example(user, assistant, path=DATA_PATH):
    rec = {"messages": [{"role": "user", "content": user},
                        {"role": "assistant", "content": assistant}]}
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    n = sum(1 for _ in open(path, encoding="utf-8"))
    print(f"Added. Dataset now has {n} examples.")

# --- Add your own (uncomment, edit, re-run) ---
# add_example("What's 2 + 2?", "2 + 2 = 4.\n\nTL;DR: 4.")

# --- Preview the current dataset ---
with open(DATA_PATH, encoding="utf-8") as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        u = rec["messages"][0]["content"]
        print(f"{i:>2}  USER: {u[:70]}")

## 12. Train on Colab, then come back

1. Open `finetune/colab_finetune.ipynb` in **Google Colab** (set runtime to **T4 GPU**).
2. Upload your `train.jsonl` when prompted, run all cells.
3. Download the produced adapter zip and unzip it into
   `models/adapters/qwen2.5-3b-tldr/` in this project.

Full instructions: `finetune/README.md`. Then run the next cell.

## 13. Run your fine-tuned model locally

Once you've trained and unzipped an adapter, this loads the base model + your adapter with `peft` and
generates with it. Until then, it prints a friendly reminder and skips.

In [ ]:
ADAPTER_DIR = "../models/adapters/qwen2.5-3b-tldr"

if not os.path.isdir(ADAPTER_DIR):
    print("No adapter yet at", ADAPTER_DIR)
    print("Train one on Colab (see finetune/README.md), unzip it there, then re-run this cell.")
else:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
    ft = PeftModel.from_pretrained(base, ADAPTER_DIR)
    ft.eval()

    msgs = [{"role": "user", "content": "What is the capital of France?"}]
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    with torch.no_grad():
        g = ft.generate(**inp, max_new_tokens=80, do_sample=False)
    print("FINE-TUNED reply:\n")
    print(tok.decode(g[0, inp["input_ids"].shape[1]:], skip_special_tokens=True))
    print("\n(Look for the trained 'TL;DR:' sign-off.)")

## 14. Compare: base vs fine-tuned, side by side

Send the **same prompt** to the base model and to your fine-tuned adapter. We reuse `ft` from the previous
cell and just toggle the LoRA adapter off (`ft.disable_adapter()`) to recover the original base behaviour
from the same weights — no second copy is loaded. Watch for the trained `TL;DR:` sign-off appearing only in
the fine-tuned answer (and on a prompt that *wasn't* in the training set, showing the style generalises).

In [ ]:
# Side-by-side: the base model vs your fine-tuned adapter, on the SAME prompt.
# We reuse `ft` from the previous cell and toggle the LoRA adapter off with
# ft.disable_adapter() to get the original base behaviour from the same weights —
# so no second copy of the model is loaded. (Two CPU generations, so give it a moment.)
import contextlib

if "ft" not in globals():
    print("Load the fine-tuned model first: run the previous cell (section 13).")
else:
    def reply(prompt, use_adapter, max_new_tokens=80):
        msgs = [{"role": "user", "content": prompt}]
        inp = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)
        ctx = contextlib.nullcontext() if use_adapter else ft.disable_adapter()
        with torch.no_grad(), ctx:
            out = ft.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
        return tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=True)

    prompt = "What is photosynthesis?"   # not in the training set -> shows the style generalises
    print("PROMPT:", prompt)
    print("\n=== BASE (adapter off) ===")
    print(reply(prompt, use_adapter=False))
    print("\n=== FINE-TUNED (adapter on) ===")
    print(reply(prompt, use_adapter=True))
    print("\n(The fine-tuned answer should end with the trained 'TL;DR:' sign-off.)")

## 15. Looking inside: token embeddings -> contextual hidden states

Sections 15-18 are a deeper look *inside* the model, reusing the `model` and `tok` you already loaded.

There are **three different things** people call "embeddings":

1. **Token embeddings** - context-*free*. The tokenizer turns a word into an integer ID; the model's first
   layer, `embed_tokens` (an `Embedding(151936, 2048)` lookup table), maps each ID to a fixed 2048-dim
   vector. Token `dog` -> the *same* vector every time, regardless of sentence.
2. **Contextual hidden states** - context-*dependent*. As that vector flows through the 36 attention+MLP
   layers, attention mixes in the surrounding tokens, so the vector at each position becomes a function of
   the whole context. This is the model's actual "thinking".
3. **Sentence embeddings** - one vector for a whole sentence (section 16).

The cell below proves #1 -> #2 with a minimal pair: `"what is a dog"` vs `"what is to dog"`, which differ
only at the 3rd token (`a` vs `to`). The final token `dog` is *identical on input* (cosine = 1.0), but watch
its vector get pulled apart layer by layer as attention lets it "see" the differing 3rd token.

In [ ]:
import torch, torch.nn.functional as F

s1, s2 = "what is a dog", "what is to dog"     # differ only at token 3: 'a' vs 'to'
i1, i2 = tok(s1, return_tensors="pt"), tok(s2, return_tensors="pt")
print("tokens 1:", tok.convert_ids_to_tokens(i1["input_ids"][0]))
print("tokens 2:", tok.convert_ids_to_tokens(i2["input_ids"][0]))

cos = lambda a, b: F.cosine_similarity(a.float(), b.float(), dim=0).item()
emb = model.get_input_embeddings()             # the embed_tokens lookup table

# (1) INPUT embedding of the shared last token 'dog' is identical -> context-free
e1, e2 = emb(i1["input_ids"])[0, -1], emb(i2["input_ids"])[0, -1]
print(f"\nINPUT embedding of 'dog', sent1 vs sent2:  cos = {cos(e1, e2):.4f}  (identical)")

# (2) Run the model and watch the 'dog' vector drift apart through the layers
with torch.no_grad():
    h1 = model(**i1, output_hidden_states=True).hidden_states   # 37 tensors: embeddings + 36 layers
    h2 = model(**i2, output_hidden_states=True).hidden_states
print("\nSame 'dog' vector, sent1 vs sent2, as it flows through the stack:")
for idx in [0, 6, 12, 18, 24, 30, 36]:
    name = "embeddings (input)" if idx == 0 else f"after layer {idx}"
    print(f"  {name:20}: cos = {cos(h1[idx][0, -1], h2[idx][0, -1]):.4f}")
print("\n-> identical at the input, pulled apart by attention to the differing 3rd token.")

## 16. Sentence embeddings & cosine similarity

To compare whole sentences by meaning (semantic search, RAG retrieval, clustering) you need **one** vector
per sentence, not one per token. The simplest recipe: **mean-pool** the contextual hidden states (average
the per-token vectors) into a single 2048-dim vector, then compare two sentences with **cosine similarity**.

We do exactly that below with Qwen. **But notice the catch:** the cosine values all bunch up high (~0.8-0.94)
and the separation is weak - a finance<->pet pair can even score higher than a within-topic pair. This is the
well-known **anisotropy** of raw decoder-LLM representations (they occupy a narrow cone of the space). It is
why production semantic search uses **dedicated embedding models** (e.g. `sentence-transformers`, or
text-embedding APIs) that are *contrastively trained* - explicitly pulling paraphrases together and pushing
unrelated text apart - so cosine similarity actually lines up with meaning. Same idea (#3 above), but a model
trained for it.

In [ ]:
import torch, torch.nn.functional as F

def sentence_embedding(text):
    ids = tok(text, return_tensors="pt")
    with torch.no_grad():
        hs = model(**ids, output_hidden_states=True).hidden_states[-1][0]   # [n_tokens, 2048]
    return hs.mean(0).float()                                               # mean-pool -> [2048]

sents = [
    "A dog is a four-legged pet.",
    "Cats and dogs are common household animals.",
    "The stock market fell sharply today.",
    "Interest rates pushed equities lower this afternoon.",
]
vecs = [sentence_embedding(s) for s in sents]
cos = lambda a, b: F.cosine_similarity(a, b, dim=0).item()

print("pairwise cosine similarity (mean-pooled last hidden state):\n")
for i in range(len(sents)):
    for j in range(i + 1, len(sents)):
        same = "  (same topic)" if (i < 2) == (j < 2) else ""
        print(f"  {cos(vecs[i], vecs[j]):+.3f}   {sents[i][:30]:32} <-> {sents[j][:30]}{same}")
print("\nNote how muddy the separation is -> this is why dedicated, contrastively-trained "
      "embedding models exist.")

## 17. One forward pass = a prediction for *every* position (and the KV cache)

When you run `model(input_ids)` on an *n*-token prompt, the output `logits` has shape **`(1, n, 151936)`** -
i.e. **one full next-token distribution per position**, not just one at the end. Row *i* means "given tokens
1..*i*, what is token *i+1*?". This works because of the **causal mask**: each position may attend only to
itself and earlier tokens, never the future - so every row is an honest next-token prediction for its prefix.

For **generation** you use only the **last** row (the prediction after the whole prompt), sample a token,
append it, and run again - the **autoregressive loop**.

Doesn't that re-process everything each step? Naively yes - but real implementations use a **KV cache**: the
keys/values for earlier tokens don't change when you append a new one (causal mask), so they are cached. The
expensive full pass happens once (the **prefill** of your prompt); each new token is then a cheap step that
computes only itself and attends over the cached past. That's why the first token of a reply lags and the
rest stream faster - and why the cache (and memory use) grows with context length.

In [ ]:
import torch

prompt = "The capital of France is Paris"
ids = tok(prompt, return_tensors="pt")["input_ids"]
with torch.no_grad():
    logits = model(ids).logits                      # [1, n, vocab]

print("input tokens :", tok.convert_ids_to_tokens(ids[0]))
print("logits shape :", tuple(logits.shape), f"-> a {logits.shape[-1]:,}-way distribution PER position\n")

top1 = logits[0].argmax(-1)                          # greedy next-token at each position
for i in range(ids.shape[1] - 1):
    ctx    = tok.decode(ids[0, : i + 1])
    pred   = tok.decode([top1[i]])
    actual = tok.decode([ids[0, i + 1]])
    hit    = "OK" if top1[i] == ids[0, i + 1] else "  "
    print(f"  {hit} after {ctx!r:34} -> predicts {pred!r:9} (actual next: {actual!r})")
print("\n-> every row predicted its next token in ONE pass; generation reuses the last row, then loops.")

## 18. Scoring "weirdness": surprisal & perplexity

Because one forward pass gives the probability the model assigned to *each actual next token*, you can score
how **surprising** a piece of text is - essentially for free, no generation needed:

- **Surprisal** of a token = `-log2 p(token | context)` -> "bits of surprise".
- **Perplexity (PPL)** = `exp(mean negative log-likelihood)` -> one headline number; low = the model finds
  the text predictable/natural, high = surprising / out-of-distribution.

This is a genuinely useful signal. Real uses: it is **the** classic language-model eval metric; **AI-text
detectors** (GLTR, GPTZero) exploit that machine text tends to have low, smooth perplexity; **jailbreak
filters** flag high-perplexity gibberish attack strings; **contamination detectors** spot benchmark items a
model has memorised (suspiciously low PPL); and in **psycholinguistics**, token surprisal predicts human
reading times.

Two caveats: it measures **typicality relative to *this* model**, not truth or quality (a true-but-rare
sentence still looks "weird"); and **tokenization** leaks in (rare words split into surprising sub-pieces).
Run the cell and compare a natural sentence with progressively weirder ones.

In [ ]:
import math, torch, torch.nn.functional as F

def perplexity(text):
    ids = tok(text, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logits = model(ids).logits[0].float()           # [n, vocab]
    logp = F.log_softmax(logits, dim=-1)
    tgt  = ids[0, 1:]                                    # the actual "next" tokens
    tlp  = logp[:-1].gather(1, tgt[:, None]).squeeze(1)  # log p the model gave each one
    bits = (-tlp / math.log(2)).tolist()                # surprisal per token
    ppl  = math.exp(-tlp.mean().item())
    return ppl, list(zip(tok.convert_ids_to_tokens(tgt), bits))

for text in [
    "The capital of France is Paris.",
    "The capital of France is asparagus.",
    "Colorless green ideas sleep furiously.",        # grammatical but meaningless (Chomsky)
    "France of the is capital Paris purple.",        # scrambled word order
]:
    ppl, pertok = perplexity(text)
    worst = max(pertok, key=lambda x: x[1])
    print(f"\nPPL = {ppl:8.1f}   {text!r}")
    print("   surprisal/token (bits):", ", ".join(f"{t}={b:.1f}" for t, b in pertok))
    print(f"   most surprising: {worst[0]!r} ({worst[1]:.1f} bits)")

## 19. Temperature: from probabilities to actual word choices

Everything so far stopped at the probability distribution. **Temperature** (`T`) controls how you *sample* a
token from it, and it enters at exactly one place - dividing the logits right before the softmax:

    p = softmax(logits / T)

- **T -> 0** (or `do_sample=False`, "greedy"): always take the single most-likely token. Deterministic and
  focused, but can be bland or repetitive. *This is what every generation cell above used*, which is why they
  were reproducible.
- **T = 1**: sample from the model's raw distribution.
- **T < 1**: sharpen it (rich get richer) -> safer, more predictable.
- **T > 1**: flatten it -> more diverse / creative / risky, and eventually incoherent.

It's usually paired with **top-p** (nucleus) or **top-k** sampling, which first restrict the choice to the
most likely tokens so a high temperature can't pick something truly absurd. The cell runs the same prompt at
several temperatures (a few CPU generations, so give it ~a minute) - watch focus turn into variety.

In [ ]:
import torch

prompt = "In one vivid sentence, describe a city at night."
inp = tok.apply_chat_template([{"role": "user", "content": prompt}],
                              add_generation_prompt=True, return_tensors="pt", return_dict=True)

def generate(temperature, seed=0, max_new_tokens=40):
    with torch.no_grad():
        if temperature == 0:                                  # greedy: temperature is ignored
            out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
        else:
            torch.manual_seed(seed)                           # makes each random draw reproducible
            out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                                  temperature=temperature, top_p=0.95)
    return tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=True)

print("temperature = 0.0  (greedy / deterministic - the setting used everywhere above)")
print("  ", generate(0.0))
for temp in [0.7, 1.3]:
    print(f"\ntemperature = {temp}  (two independent samples - watch them diverge)")
    print("  A:", generate(temp, seed=0))
    print("  B:", generate(temp, seed=1))
print("\n-> low T: focused & repeatable.  high T: more variety, less safe.  Try editing the prompt/temps.")

## 20. Where to go next

- **Sampling**: try `do_sample=True, temperature=0.7, top_p=0.9` in `generate` and watch variety change.
- **Bigger context**: feed a paragraph and ask for a summary.
- **More data**: add 30-50 consistent examples and re-train — the style effect gets much stronger.
- **Quantization**: compare `transformers` (bf16) vs Ollama (4-bit) speed and quality.
- **Read the source**: `transformers` model code for Qwen lives in your `.venv` under
  `transformers/models/qwen2/` — open `modeling_qwen2.py` to see attention + MLP implemented.